In [ ]:
import numpy as np
import xarray as xr
from matplotlib import pyplot as plt
import pandas as pd
import glob
import os


In [ ]:
# ----------------------------------------------------------------------------
# Paths
# ----------------------------------------------------------------------------
path_ensemble="../../ensemble_reduced/*"
path_output="../output"
sim_paths = sorted(glob.glob(path_ensemble))
file_name = "yelmo2D.nc" 
regs=xr.open_dataset("/p/projects/megarun/luciagu/data/lauritzen2025/Greenland_Basins_PS_v1.4.2_8km.nc")

topo = xr.open_dataset("/p/projects/megarun/ice_data/Greenland/GRL-8KM/GRL-8KM_TOPO-M17.nc")


In [80]:
def variables(sim,mask,basin):

    time_fun = sim.time * 1e-3
    if basin=="all":
        h=xr.where((sim.f_grnd>0)&(sim.H_ice>0),sim.H_ice,0)
    else:
        h=xr.where((sim.f_grnd>0)&(sim.H_ice>0)&(mask==basin),sim.H_ice,0)

    A = (h != 0).sum(dim=("yc", "xc")) * 8 * 8 * 1e-6 
    V = h.sum(dim=("yc", "xc")) * 1e-3 * 8 * 8 * 1e-6
    return time_fun, A, V

region_ids = np.unique(regs.mask)

In [ ]:

T_ann_ensemble = []
T_sum_ensemble = []
A_ensemble = []   
valid_sim_indices = []   

for i, sim_path in enumerate(sim_paths):

    file_path = os.path.join(sim_path, file_name)

    try:
        yelmo = xr.open_dataset(file_path)
    except FileNotFoundError:
        print(f"[ERROR] No se encontró el archivo: {file_path}. Se omite esta simulación.")
        continue
    except Exception as e:
        print(f"[ERROR] No se pudo abrir {file_path}: {e}. Se omite esta simulación.")
        continue

    time=yelmo.time.values
    T_ann=(yelmo.Ta_ann.where(yelmo.mask_bed==0)).mean(dim=["xc","yc"])
    T_sum=(yelmo.Ta_sum.where(yelmo.mask_bed==0)).mean(dim=["xc","yc"])
    
    T_ann_ensemble.append(T_ann)
    T_sum_ensemble.append(T_sum)
    
    A_all = []
    for r in region_ids:
        _, A, _ = variables(yelmo, regs.mask, r)
        A_all.append(A.values)

    # convertir lista → array (tiempo x regiones)
    A_all = np.stack(A_all, axis=1)
    A_ensemble.append(A_all)
    valid_sim_indices.append(i)   


# convertir lista → array (simulación x tiempo)
T_ann_ensemble = np.stack(T_ann_ensemble, axis=0)
T_sum_ensemble = np.stack(T_sum_ensemble, axis=0)
A_ensemble = np.stack(A_ensemble, axis=0)

# construir dataset final
ds_ensemble = xr.Dataset(
    data_vars={
        "T_ann": (("sim", "time"), T_ann_ensemble),
        "T_sum": (("sim", "time"), T_sum_ensemble),
        "A": (("sim", "time", "region"), A_ensemble)
    },
    coords={
        "sim": np.array(valid_sim_indices),
        "time": time,
        "region": region_ids
    }
)



[ERROR] No se encontró el archivo: /p/projects/megarun/luciagu/data/tabone2024/ensemble/1670/yelmo2D.nc. Se omite esta simulación.
[ERROR] No se encontró el archivo: /p/projects/megarun/luciagu/data/tabone2024/ensemble/461/yelmo2D.nc. Se omite esta simulación.
[ERROR] No se encontró el archivo: /p/projects/megarun/luciagu/data/tabone2024/ensemble/462/yelmo2D.nc. Se omite esta simulación.
[ERROR] No se encontró el archivo: /p/projects/megarun/luciagu/data/tabone2024/ensemble/463/yelmo2D.nc. Se omite esta simulación.
[ERROR] No se encontró el archivo: /p/projects/megarun/luciagu/data/tabone2024/ensemble/464/yelmo2D.nc. Se omite esta simulación.
[ERROR] No se encontró el archivo: /p/projects/megarun/luciagu/data/tabone2024/ensemble/465/yelmo2D.nc. Se omite esta simulación.
[ERROR] No se encontró el archivo: /p/projects/megarun/luciagu/data/tabone2024/ensemble/466/yelmo2D.nc. Se omite esta simulación.
[ERROR] No se encontró el archivo: /p/projects/megarun/luciagu/data/tabone2024/ensemble/4

/home/luciagu/.local/lib/python3.12/site-packages/xarray/backends/api.py:668: RuntimeWarning: 'netcdf4' fails while guessing
  engine = plugins.guess_engine(filename_or_obj)
/home/luciagu/.local/lib/python3.12/site-packages/xarray/backends/api.py:668: RuntimeWarning: 'h5netcdf' fails while guessing
  engine = plugins.guess_engine(filename_or_obj)
/home/luciagu/.local/lib/python3.12/site-packages/xarray/backends/api.py:668: RuntimeWarning: 'scipy' fails while guessing
  engine = plugins.guess_engine(filename_or_obj)


In [85]:
ds_ensemble.to_netcdf("../output/timeseries_from2D_part1.nc")

In [ ]:
# ---------------------------------------------
# Calculo area por regiones en PaleoGrIS repetr
# ---------------------------------------------

tiempos = np.array([0., -7000., -7500., -8000., -8500., -9000., -9500.,
                    -10000., -10500., -11000., -11500., -12250., -13000.])

area_celda = 2 * 2 * 1e-6  # millones de km²
# --- Calcular área cubierta total y por región ---
region_ids = np.arange(1, 9)  # regiones 1–9
area_por_region = {r: [] for r in region_ids}
area_total = []

for t in tiempos:
    cubierta = xr.where(-leg.ic >= t, 1, 0)  
    
    # área total
    A_t = cubierta.sum() * area_celda
    area_total.append(A_t.item())
    
    # área por región
    for r in region_ids:
        region_mask = (mask_bed_hi == r)
        A_r_t = cubierta.where(region_mask).sum() * area_celda
        area_por_region[r].append(A_r_t.item())

data_vars = {"total": (["time"], area_total)}
for r in region_ids:
    data_vars[f"region_{r}"] = (["time"], area_por_region[r])

serie_area = xr.Dataset(
    data_vars=data_vars,
    coords={"time": tiempos},
)
# serie_area.to_netcdf("/home/luciagu/data/leger2024/serie_area_tmax2.nc")
